In [38]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
import pandas as pd
import numpy as np

In [39]:
# Load the new dataset
file_path = '/content/drive/MyDrive/Bangkit/Capstone/Online Sales Data.csv'
data = pd.read_csv(file_path)

In [40]:
# Display column names and first few rows to understand structure
print(data.columns)

Index(['Transaction ID', 'Date', 'Product Category', 'Product Name',
       'Units Sold', 'Unit Price', 'Total Revenue', 'Region',
       'Payment Method'],
      dtype='object')


In [41]:
print(data.head())

   Transaction ID        Date Product Category             Product Name  \
0           10001  2024-01-01      Electronics            iPhone 14 Pro   
1           10002  2024-01-02  Home Appliances         Dyson V11 Vacuum   
2           10003  2024-01-03         Clothing         Levi's 501 Jeans   
3           10004  2024-01-04            Books        The Da Vinci Code   
4           10005  2024-01-05  Beauty Products  Neutrogena Skincare Set   

   Units Sold  Unit Price  Total Revenue         Region Payment Method  
0           2      999.99        1999.98  North America    Credit Card  
1           1      499.99         499.99         Europe         PayPal  
2           3       69.99         209.97           Asia     Debit Card  
3           4       15.99          63.96  North America    Credit Card  
4           1       89.99          89.99         Europe         PayPal  


In [42]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    240 non-null    int64  
 1   Date              240 non-null    object 
 2   Product Category  240 non-null    object 
 3   Product Name      240 non-null    object 
 4   Units Sold        240 non-null    int64  
 5   Unit Price        240 non-null    float64
 6   Total Revenue     240 non-null    float64
 7   Region            240 non-null    object 
 8   Payment Method    240 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 17.0+ KB
None


In [43]:
# Preprocess Data
data = data[['Date', 'Total Revenue']]  # Select relevant columns
data['Date'] = pd.to_datetime(data['Date'])  # Convert to datetime
data = data.sort_values('Date')  # Sort by date
data.set_index('Date', inplace=True)  # Set date as index

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Create sequences
def create_sequences(data, sequence_length):
    x, y = [], []
    for i in range(len(data) - sequence_length):
        x.append(data[i:i + sequence_length])
        y.append(data[i + sequence_length])
    return np.array(x), np.array(y)

sequence_length = 30  # Use past 30 days for prediction
x, y = create_sequences(scaled_data, sequence_length)

# Split into train and test sets
split = int(len(x) * 0.8)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

In [44]:
# Build LSTM Model
model = Sequential([
    tf.keras.layers.LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    tf.keras.layers.LSTM(50, return_sequences=False),
    tf.keras.layers.Dense(units=25),
    tf.keras.layers.Dense(units=1)
])

# compile model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the Model
model.fit(x_train, y_train, batch_size=32, epochs=20, validation_data=(x_test, y_test))

Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - loss: 0.0244 - val_loss: 0.0112
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - loss: 0.0131 - val_loss: 0.0104
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.0182 - val_loss: 0.0104
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0137 - val_loss: 0.0097
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.0140 - val_loss: 0.0096
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.0122 - val_loss: 0.0098
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - loss: 0.0160 - val_loss: 0.0108
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.0194 - val_loss: 0.0100
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.0120 - val_loss: 0.0097
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0134 - val_loss: 0.0099
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.0161 - val_loss: 0.0100
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0140 - val_loss: 0.0103
Epoch 13/20

In [45]:
# Make Predictions
predicted = model.predict(x_test)
predicted = scaler.inverse_transform(predicted)  # Rescale to original values
actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Predict Future Value
last_sequence = scaled_data[-sequence_length:].reshape(1, sequence_length, 1)
future_prediction = model.predict(last_sequence)
future_value = scaler.inverse_transform(future_prediction)

print(f"Prediksi H+1 Setelah Tanggal Terakhir di Dataset: {future_value[0][0]}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 300ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Prediksi H+1 Setelah Tanggal Terakhir di Dataset: 340.6508483886719
